# 🎓 Smart University Assistant — Integrated AI Project

**Goal:** A single student-support assistant that answers questions about courses and university procedures, checks sample registration records, remembers recent conversation context, and evaluates its own responses.

**Integrated lab ideas:** tokenization and cost, generation, structured tools, document processing and RAG, conversation memory, caching, guardrails, tracing, and evaluation.

> **Data disclaimer:** All course descriptions, registration statuses, office locations and instructions below are **illustrative demo data**, not verified policies of a real university. This notebook **reads** sample registration data; it does not submit or modify real registrations.

**Workflow:** Student question → safety check → intent/tool routing or document retrieval → optional local LLM → evidence-grounded answer → cache/history/trace → evaluation.

## 1. Setup
Run cells in order. RAG uses sentence embeddings and a Chroma vector collection. The optional language model is downloaded **only if enabled**; the complete assistant works in evidence-only mode by default.

In [ ]:
%pip -q install "sentence-transformers>=3" "chromadb>=0.5" "pydantic>=2" "tiktoken>=0.7" "transformers>=4.45" torch matplotlib

In [ ]:
import re, json, time, sqlite3, hashlib
from dataclasses import dataclass
from collections import Counter
from functools import wraps
from pydantic import BaseModel, Field, ValidationError

ASSISTANT_NAME = "Smart University Assistant"
USE_LOCAL_LLM = False  # Set True to load LiquidAI/LFM2.5-350M for grounded paraphrasing
MAX_HISTORY = 6
DB_PATH = "university_assistant_demo.db"
print("🎓", ASSISTANT_NAME, "| Mode:", "local LLM" if USE_LOCAL_LLM else "evidence-only")

## 2. University knowledge and student records
We use a *small, explicitly fictional* knowledge base so every answer can be checked. Replace it with verified university documentation and authorized systems before real use.

In [ ]:
KNOWLEDGE = [
    {"id": "course_ml", "topic": "courses", "title": "Machine Learning", "text": "Machine Learning introduces predictive models, supervised learning, and model evaluation."},
    {"id": "course_ds", "topic": "courses", "title": "Data Science", "text": "Data Science covers data analysis, visualization, and introductory statistics."},
    {"id": "course_ai", "topic": "courses", "title": "Artificial Intelligence", "text": "Artificial Intelligence explores intelligent systems, search techniques, and AI applications."},
    {"id": "registration", "topic": "procedure", "title": "Course registration", "text": "In this demonstration, students review available courses, then consult the student portal for official registration steps. The assistant cannot enroll students."},
    {"id": "exam_schedule", "topic": "procedure", "title": "Exam schedule", "text": "For an official examination date or venue, check the student portal or ask the academic department. No live timetable is connected to this assistant."},
    {"id": "department", "topic": "location", "title": "Computer Science department", "text": "The exact building and room of the Computer Science department are not included in this demo knowledge base. Contact the university for a confirmed location."},
]
# Example records only — not live student data.
REGISTRATIONS = {
    ("ahmed", "machine learning"): "Confirmed (demo record)",
    ("sara", "data science"): "Confirmed (demo record)",
}
for doc in KNOWLEDGE: print(f"{doc['id']}: {doc['title']}")

## 3. University tools
Validate requests and use simple tools to retrieve sample course and registration information.

In [ ]:
class RegistrationRequest(BaseModel):
    student_name: str = Field(min_length=2)
    course: str = Field(min_length=2)

def list_courses():
    return [d["title"] for d in KNOWLEDGE if d["topic"] == "courses"]

def check_registration(student_name: str, course: str):
    query = RegistrationRequest(student_name=student_name.strip(), course=course.strip())
    status = REGISTRATIONS.get((query.student_name.lower(), query.course.lower()))
    return (f"Demo registration for {query.student_name} in {query.course}: {status}."
            if status else "No matching registration in the demo records. This does not confirm official status.")

TOOLS = {"list_courses": list_courses, "check_registration": check_registration}
print("Available tools:", list(TOOLS))
print(check_registration("Ahmed", "Machine Learning"))

## 4. University knowledge retrieval
Create an embedding index for the sample university knowledge base and attach sources to answers.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)
texts = [f"{d['title']}. {d['text']}" for d in KNOWLEDGE]
vectors = embedder.encode(texts, normalize_embeddings=True).tolist()
client = chromadb.Client()
collection = client.get_or_create_collection("university_assistant_demo")
collection.upsert(ids=[d['id'] for d in KNOWLEDGE], documents=texts,
                  embeddings=vectors, metadatas=[{"topic": d['topic'], "title": d['title']} for d in KNOWLEDGE])

def retrieve(question: str, k: int = 2):
    v = embedder.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=v, n_results=min(k, len(KNOWLEDGE)),
                               include=["documents", "metadatas", "distances"])
    return [{"id": i, "text": t, "metadata": m, "distance": dist}
            for i, t, m, dist in zip(results["ids"][0], results["documents"][0],
                                     results["metadatas"][0], results["distances"][0])]

for item in retrieve("What does Machine Learning cover?"):
    print(item["id"], "→", item["text"])

## 5. Optional local language model
The assistant works without the model. Set `USE_LOCAL_LLM = True` to try the LiquidAI model for rephrasing retrieved information. Model-generated responses may be inaccurate.

In [ ]:
LOCAL_MODEL_ID = "LiquidAI/LFM2.5-350M"
local_tokenizer, local_model = None, None
if USE_LOCAL_LLM:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID, trust_remote_code=True)
    local_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_ID, trust_remote_code=True,
                                                      torch_dtype="auto", device_map="auto")
    local_model.eval()
    print("Loaded:", LOCAL_MODEL_ID)
else:
    print("LLM skipped; evidence-only answers available immediately after embeddings are ready.")

def llm_paraphrase(question: str, evidence: str) -> str:
    messages = [
        {"role": "system", "content": ("You are a university assistant. Use ONLY the supplied evidence. "
          "Do not invent dates, times, course prerequisites, policies, or enrollments. "
          "If the evidence does not answer the question, say that the information is unconfirmed.")},
        {"role": "user", "content": f"Evidence:\n{evidence}\n\nQuestion: {question}"},
    ]
    inputs = local_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                                  return_tensors="pt", return_dict=True)
    inputs = {key: value.to(local_model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        generated = local_model.generate(**inputs, max_new_tokens=110, do_sample=False,
                                         pad_token_id=local_tokenizer.eos_token_id)
    return local_tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:],
                                  skip_special_tokens=True).strip()

## 6. Conversation memory and safety
Save conversations in SQLite, cache repeated questions, and keep a simple trace of assistant requests.

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.execute("CREATE TABLE IF NOT EXISTS chat (id INTEGER PRIMARY KEY AUTOINCREMENT, session TEXT, question TEXT, answer TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)")
conn.commit()
CACHE = {}
TRACE = []

BLOCKED_PATTERNS = [r"\b(password|secret key|api key|private student record)\b"]
def safety_check(question):
    if any(re.search(pattern, question, re.I) for pattern in BLOCKED_PATTERNS):
        return "I cannot provide passwords, secret keys, or another student's private records."
    return None

def get_history(session="demo", limit=MAX_HISTORY):
    rows = conn.execute("SELECT question, answer FROM chat WHERE session=? ORDER BY id DESC LIMIT ?", (session, limit)).fetchall()
    return list(reversed(rows))

def store_turn(session, question, answer):
    conn.execute("INSERT INTO chat (session, question, answer) VALUES (?, ?, ?)", (session, question, answer))
    conn.commit()

def trace_event(question, route, elapsed, cache_hit, sources):
    TRACE.append({"question": question, "route": route, "latency_s": round(elapsed, 3),
                  "cache_hit": cache_hit, "sources": sources})

## 7. One assistant pipeline
Route known course-list and registration requests to structured tools, and general knowledge questions to RAG. Unknown or unsupported requests should not receive made-up university facts.

In [ ]:
COURSE_ALIASES = {"machine learning": "Machine Learning", "data science": "Data Science",
                  "artificial intelligence": "Artificial Intelligence"}

def route_question(question: str):
    q = question.lower()
    if ("list" in q or "available" in q or "what courses" in q) and "course" in q:
        return "list_courses"
    if ("registration status" in q or "registered" in q or "am i enrolled" in q or "check my registration" in q):
        return "check_registration"
    return "rag"

def extract_course(question: str):
    q = question.lower()
    return next((title for alias, title in COURSE_ALIASES.items() if alias in q), None)

def answer_question(question: str, session="demo", student_name=None, use_cache=True):
    question = question.strip()
    if not question:
        return "Please enter a university-related question."
    start = time.perf_counter()
    route = route_question(question)
    cache_key = (question.lower(), (student_name or "").lower(), USE_LOCAL_LLM)
    blocked = safety_check(question)
    cached = False
    sources = []
    if blocked:
        answer, route = blocked, "safety"
    elif use_cache and cache_key in CACHE:
        answer = CACHE[cache_key]
        cached = True
    elif route == "list_courses":
        answer = "Available demo courses: " + ", ".join(list_courses()) + "."
        sources = ["demo course catalog"]
    elif route == "check_registration":
        course = extract_course(question)
        if not student_name or not course:
            answer = "To check a demo registration, provide your student name and the exact course name."
        else:
            answer = check_registration(student_name, course)
        sources = ["demo registrations"]
    else:
        matches = retrieve(question, k=2)
        # Only answer when there is a clear topic match or high cosine similarity;
        # do not treat the closest document as evidence for every question.
        best = matches[0]
        similarity = 1 - best["distance"]  # cosine distance with normalized embeddings
        topic_match = any(word in question.lower() for word in
                          re.findall(r"[a-z]+", best["metadata"]["title"].lower()) if len(word) > 3)
        if similarity < 0.40 and not topic_match:
            answer = "I don't have confirmed information about that in the demo university knowledge base. Please check the official university portal."
        else:
            sources = [best["id"]]
            if USE_LOCAL_LLM:
                try:
                    answer = llm_paraphrase(question, best["text"])
                    answer += f"\n[Generated from demo source: {best['id']}; verify before use]"
                except Exception:
                    answer = best["text"] + f" [Demo source: {best['id']}]"
            else:
                answer = best["text"] + f" [Demo source: {best['id']}]"
    if not blocked and not cached:
        CACHE[cache_key] = answer
    store_turn(session, question, answer)
    trace_event(question, route, time.perf_counter()-start, cached, sources)
    return answer

print(answer_question("What does Machine Learning cover?"))

## 8. Student demo (full integrated workflow)
These queries exercise tool routing, document retrieval, unknown-information handling and caching.

In [ ]:
demo_questions = [
    ("What courses are available?", None),
    ("What does Data Science cover?", None),
    ("Check my registration status for Machine Learning", "Ahmed"),
    ("Where do I find my official exam schedule?", None),
    ("What is the confirmed tuition fee for next semester?", None),
    ("What does Data Science cover?", None),  # cache hit
]
for question, name in demo_questions:
    print("\n👤", question)
    print("🤖", answer_question(question, session="demo_run", student_name=name))
print("\nRecent conversations:")
for q, a in get_history("demo_run", 3): print("-", q, "→", a[:110])
print("\nLast trace:", TRACE[-1])

## 9. Tokenization and cost estimation
Estimate token usage and illustrative API costs. These are examples, not verified current provider prices.

In [ ]:
import tiktoken
encoder = tiktoken.get_encoding("cl100k_base")
example_question = "How can I register for a machine learning course?"
example_context = KNOWLEDGE[3]["text"]
input_tokens = len(encoder.encode(example_question + "\n" + example_context))
estimated_output_tokens = 80
# Set your own verified provider price assumptions, USD per 1M tokens:
example_input_rate, example_output_rate = 1.00, 4.00
estimated_usd = input_tokens * example_input_rate / 1_000_000 + estimated_output_tokens * example_output_rate / 1_000_000
print("Input tokens:", input_tokens)
print("Assumed output tokens:", estimated_output_tokens)
print(f"Illustrative cloud cost: ${estimated_usd:.6f}")
print("English tokens:", len(encoder.encode("How can I register?")))
print("Arabic tokens:", len(encoder.encode("كيف أسجل في المقرر؟")))

## 10. Evaluation
Check sample questions and responses against a small test set. Passing simple keyword checks is not a guarantee of correctness.

In [ ]:
TESTS = [
    {"question": "What courses are available?", "expected": "Machine Learning", "route": "list_courses"},
    {"question": "What does Machine Learning cover?", "expected": "predictive models", "route": "rag"},
    {"question": "What does Data Science cover?", "expected": "visualization", "route": "rag"},
    {"question": "Check my registration status for Machine Learning", "expected": "Confirmed", "route": "check_registration", "student_name": "Ahmed"},
    {"question": "Where can I check my official exam schedule?", "expected": "student portal", "route": "rag"},
]
results = []
for test in TESTS:
    response = answer_question(test["question"], session="evaluation",
                               student_name=test.get("student_name"), use_cache=False)
    passed = test["expected"].lower() in response.lower() and route_question(test["question"]) == test["route"]
    results.append(passed)
    print(("PASS ✅" if passed else "FAIL ❌"), test["question"], "→", response)
print(f"\nGolden-set pass rate: {sum(results)}/{len(results)} = {sum(results)/len(results):.0%}")
print("Total traced calls:", len(TRACE), "| Cache hits:", sum(x["cache_hit"] for x in TRACE))

## 11. Try your own questions
Change the values and run this cell again. For a demo registration query, specify `student_name` separately.

In [ ]:
my_question = "Which course focuses on intelligent systems?"
my_student_name = None
print("👤", my_question)
print("🤖", answer_question(my_question, session="my_session", student_name=my_student_name))

## What this project demonstrates

The **Smart University Assistant** is an AI-powered prototype that helps students explore university information through natural language questions.

- **Natural language processing:** Understanding and responding to student questions.
- **Language models:** Optional local generation for more natural responses.
- **University tools:** Listing courses and checking simulated registration information.
- **Retrieval-augmented generation (RAG):** Finding relevant university information and citing demo sources.
- **Conversation memory:** Saving recent interactions locally for review.
- **Safety and evaluation:** Basic input checks, request tracing, and response tests.
- **Token and cost analysis:** Estimating token usage and illustrative inference costs.

**Limitations:** This project uses invented university information and simulated registrations. It does not connect to a real university system, and AI-generated answers may be inaccurate. Do not enter real student identifiers or sensitive data into the demo.
